## 1. SVM regression — K-fold with NAVG & KEEP_LAST_PERCENT


## 2. Imports & Global Config

In [2]:
# =========================
# Core Python
# =========================
import os
import sys
import json
import csv
import logging
from collections import defaultdict
from typing import List, Tuple, Dict, Any

# =========================
# Numerical & Data Handling
# =========================
import numpy as np
import pandas as pd

# =========================
# Machine Learning
# =========================
from sklearn.svm import SVR
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    matthews_corrcoef
)

# =========================
# Statistics
# =========================
from scipy.stats import pearsonr

# =========================
# Model Persistence
# =========================
import joblib

# =========================
# Reproducibility
# =========================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 3. Load Experimental & MD Data, Merge on Sequence

Assumes:
- `exp_data_all.csv` contains at least: `sequence`, `bind_avg`  
- `rawdat.csv` contains MD features plus `sequence` and `run`  


In [3]:
# ID and label columns
id_col    = "sequence"
label_col = "bind_avg"

# --- Experimental data ---
df_exp = pd.read_csv("../Data/exp_data_all.csv")
ref_data = df_exp[[id_col, label_col]].copy()
print("Experimental data:", ref_data.shape)
display(ref_data.head())

# --- MD / feature data ---
usecols = [
    "sequence", "run",
    "VDWAALS", "EEL", "EGB", "ESURF",
    "HB Energy", "Hydrophobic Energy", "Pi-Pi Energy",
    "Delta_Entropy"
]

Experimental data: (168, 2)


,sequence,bind_avg
0,GAGGAAGCAGCCCTCGCCCCTGTCGGTGGAAAGAAG,-0.758634
1,GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG,-1.003319
2,ATCTGATCAAAACAACGAATTCCAAAACAAAGTAAT,-0.800322
3,CCAATATTCCTTTGTGAGACCCTCCACAAATGCTAA,-0.941242
4,GAGGACGCGAACCGGCACGCTGCGCCTTTAAGGAGT,-0.684116


In [4]:
df_md = pd.read_csv("../Data/rawdat.csv", usecols=usecols)
feature_data = df_md.copy()
print("Feature data:", feature_data.shape)
display(feature_data.head())


Feature data: (272160, 10)


,sequence,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy
0,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-252.110,-1886.830,1841.253,-36.482,-1.940432,-165.447020,-1.655191e-03,-26.046553
1,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-238.510,-1881.424,1835.847,-36.023,-2.003962,-155.422935,-4.708262e-02,-24.150637
2,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-246.721,-1895.687,1851.589,-35.802,-2.269901,-142.386371,-5.901517e-29,-24.329875
3,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-235.671,-1857.573,1814.002,-34.799,-2.838678,-147.918585,-3.236084e-07,-23.615145
4,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-230.214,-1897.268,1847.934,-34.391,-2.810414,-151.012478,-1.784784e-05,-23.698348


In [5]:

# --- Merge on sequence ---
df_merged = pd.merge(feature_data, ref_data, on=id_col, how="inner")
print("Merged data:", df_merged.shape)
display(df_merged.head())

# Shuffle merged rows
df_merged = df_merged.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print("Merged (shuffled):", df_merged.shape)

Merged data: (272160, 11)


,sequence,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy,bind_avg
0,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-252.110,-1886.830,1841.253,-36.482,-1.940432,-165.447020,-1.655191e-03,-26.046553,1.531336
1,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-238.510,-1881.424,1835.847,-36.023,-2.003962,-155.422935,-4.708262e-02,-24.150637,1.531336
2,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-246.721,-1895.687,1851.589,-35.802,-2.269901,-142.386371,-5.901517e-29,-24.329875,1.531336
3,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-235.671,-1857.573,1814.002,-34.799,-2.838678,-147.918585,-3.236084e-07,-23.615145,1.531336
4,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-230.214,-1897.268,1847.934,-34.391,-2.810414,-151.012478,-1.784784e-05,-23.698348,1.531336


Merged (shuffled): (272160, 11)


## 4. Initial Train/Test Split by Sequence (80/20)

We split at the **sequence level** so that sequences in test are not seen in training.


In [6]:
test_percentage = 0.20

unique_seqs = df_merged[id_col].unique()
np.random.seed(RANDOM_STATE)
np.random.shuffle(unique_seqs)

n_train = int((1.0 - test_percentage) * len(unique_seqs))

train_seqs = unique_seqs[:n_train]
test_seqs  = unique_seqs[n_train:]

df_train = df_merged[df_merged[id_col].isin(train_seqs)].copy()
df_test  = df_merged[df_merged[id_col].isin(test_seqs)].copy()

print("Train shape:", df_train.shape)
print("Test  shape:", df_test.shape)

# Save initial split (optional, for traceability)
df_train.to_csv("reg_trn_final.csv", index=False)
df_test.to_csv("reg_tst_preprocess.csv", index=False)

Train shape: (217080, 11)
Test  shape: (55080, 11)


## 5. KEEP_LAST_PERCENT = 50

For each sequence, keep only the last 50% of rows (e.g., later frames / runs).


In [7]:
KEEP_LAST_PERCENT = 100  # from hyperparameter search

df_sorted = df_train.copy()

group_sizes = df_sorted.groupby(id_col)[id_col].transform("size")
cumcount    = df_sorted.groupby(id_col).cumcount()

n_keep = (group_sizes * (KEEP_LAST_PERCENT / 100.0)).astype(int)
n_keep = n_keep.mask(n_keep < 1, 1)  # at least 1 row if fraction > 0

mask = cumcount >= (group_sizes - n_keep)
df_train = df_sorted[mask].reset_index(drop=True)

print("After KEEP_LAST_PERCENT:", df_train.shape)
display(df_train.head())

After KEEP_LAST_PERCENT: (217080, 11)


,sequence,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy,bind_avg
0,CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT,7,-207.264,-1958.570,1908.590,-34.442,-3.478696,-140.468722,-5.839134,-22.871906,1.976320
1,TTAGAAAAATAGTTTAAAATCTAGAGTTAATTAACC,3,-197.506,-1830.441,1787.958,-29.223,-2.514019,-120.507243,-4.043271,-18.798981,0.002694
2,CCAGCTCTCCACCGCCGCGTGCGCCTGCAGACGCTC,1,-207.553,-1891.794,1845.707,-31.909,-13.343306,-138.470452,-3.535617,-22.011054,0.153350
3,CCCCCAGCGCTCCGGCACGCGCCGGGAGACCTCCGG,19,-201.484,-1923.635,1874.505,-31.059,-4.628190,-138.103395,-0.006414,-22.886912,-0.483023
4,TCCGCCTCCGTCCCCCACGTTGCGTTCTGGGAGTTG,11,-203.541,-1924.910,1872.754,-32.963,-3.595719,-144.264637,-4.272515,-23.900001,0.033983


## 6. NAVG = 20 — Average Numeric Features Per Sequence

In [9]:
NAVG = 20  # from hyperparameter search

def average_features_for_sequence(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    """Shuffle a sequence's rows, chop into chunks of size `navg`, and average numeric features."""
    if df.empty:
        return pd.DataFrame(columns=df.columns)

    df_shuf = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    n_rows = len(df_shuf)
    chunks = []

    numeric_cols = df_shuf.select_dtypes(include=[np.number]).columns.tolist()
    if id_col in numeric_cols:
        numeric_cols.remove(id_col)
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)

    for start in range(0, n_rows, navg):
        end = min(start + navg, n_rows)
        sub = df_shuf.iloc[start:end]

        row = {}
        row[id_col]    = sub[id_col].iloc[0]
        row[label_col] = sub[label_col].iloc[0]
        for col in numeric_cols:
            row[col] = sub[col].mean()

        chunks.append(row)

    return pd.DataFrame(chunks)

def average_features_for_mutants(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    all_chunks = []
    for seq, group in df.groupby(id_col):
        chunk_df = average_features_for_sequence(group, navg, id_col, label_col, random_state)
        all_chunks.append(chunk_df)
    if not all_chunks:
        return pd.DataFrame(columns=df.columns)
    return pd.concat(all_chunks, ignore_index=True)

df_train_avg = average_features_for_mutants(
    df_train,
    navg=NAVG,
    id_col=id_col,
    label_col=label_col,
    random_state=random_state
)

print("After NAVG averaging:", df_train_avg.shape)
display(df_train_avg.head())

After NAVG averaging: (10854, 11)


,sequence,bind_avg,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy
0,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,11.15,-216.04365,-1889.25845,1842.90880,-32.95365,-13.547972,-145.302473,-2.042028,-23.422286
1,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,10.45,-211.11960,-1885.08835,1839.20195,-31.94215,-14.572665,-139.682198,-2.487052,-22.274451
2,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,10.65,-208.69625,-1889.28055,1841.86925,-31.79340,-14.453291,-139.509590,-2.803277,-22.675960
3,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,11.20,-205.71785,-1878.70740,1832.68840,-31.13805,-13.350216,-137.357618,-1.799540,-22.172318
4,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,10.65,-215.72575,-1895.92670,1848.62760,-32.89245,-13.314758,-142.652436,-2.941463,-22.951100


## 7. Compute Mean/Std, Save, and Standardize

Compute Z-score stats on training (averaged) data and apply them to get
a standardized training table.


In [6]:
from typing import Tuple

def compute_mean_std(
    df: pd.DataFrame,
    id_col: str,
    label_col: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_col in numeric_cols:
        numeric_cols.remove(id_col)
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)
    means = [(c, df[c].mean()) for c in numeric_cols]
    stds  = [(c, df[c].std())  for c in numeric_cols]
    return (
        pd.DataFrame(means, columns=["colname", "mean"]),
        pd.DataFrame(stds,  columns=["colname", "std"])
    )

def save_mean_std(mean_df: pd.DataFrame, std_df: pd.DataFrame, filename: str) -> None:
    merged = pd.merge(mean_df, std_df, on="colname")
    merged.to_csv(filename, index=False)

def load_mean_std(filename: str) -> Dict[str, Tuple[float, float]]:
    df = pd.read_csv(filename)
    stats = {}
    for _, row in df.iterrows():
        stats[row["colname"]] = (row["mean"], row["std"])
    return stats

def apply_standardization(
    df: pd.DataFrame,
    stats: Dict[str, Tuple[float, float]],
    id_col: str,
    label_col: str
) -> pd.DataFrame:
    df_std = df.copy()
    for col, (m, s) in stats.items():
        if col in df_std.columns and s not in (0, None) and not np.isnan(s):
            df_std[col] = (df_std[col] - m) / s
    return df_std

# Compute and save stats
mean_df, std_df = compute_mean_std(df_train_avg, id_col, label_col)
save_mean_std(mean_df, std_df, "reg_train_mean_std.csv")

# Apply standardization
stats_dict = load_mean_std("reg_train_mean_std.csv")
df_train_std = apply_standardization(df_train_avg, stats_dict, id_col, label_col)

print("Standardized training data:", df_train_std.shape)
display(df_train_std.head())

Standardized training data: (402, 11)


,sequence,bind_avg,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy
0,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,-2.797853,-0.350663,0.891499,-0.828914,-0.099992,-0.993413,-0.078806,0.296857,-0.547173
1,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,2.594449,-0.312523,1.100077,-1.022562,0.025180,-1.004207,-0.043614,0.621551,-0.449631
2,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,1.172119,-0.250047,-0.484288,1.389218,-1.321073,-0.004547,-0.948965,-0.111224,0.464743,-0.528115
3,AACCTAAAACAAAAAATCTATTATCTGATTGCAGGT,-0.378805,-0.306473,0.614518,-2.202065,2.311362,0.225309,0.839205,0.450586,-1.719587,0.584161
4,AACCTAAAACAAAAAATCTATTATCTGATTGCAGGT,-0.378805,1.616099,0.493766,-2.316428,2.439430,-0.048608,0.836292,0.267677,-2.178969,0.524179


## 8. Repeated Group K-Fold (on Sequence IDs)

We now create:
- `reg_trn_{repeat}_{fold}.csv`  
- `reg_val_{repeat}_{fold}.csv`  
for cross-validation.


In [7]:
num_repeats = 3
kfold       = 5

groups = df_train_std[id_col]

for repeat_idx in range(num_repeats):
    repeat_seed = random_state + 100 * repeat_idx
    np.random.seed(repeat_seed)

    kf = GroupKFold(n_splits=kfold)
    split_iter = kf.split(df_train_std, groups=groups)

    fold_counter = 0
    for trn_idx, val_idx in split_iter:
        df_fold_trn = df_train_std.iloc[trn_idx].copy()
        df_fold_val = df_train_std.iloc[val_idx].copy()

        # Ensure ID is first column
        col_order = [id_col] + [c for c in df_fold_trn.columns if c != id_col]
        df_fold_trn = df_fold_trn[col_order]
        df_fold_val = df_fold_val[col_order]

        fold_train_csv = f"reg_trn_{repeat_idx}_{fold_counter}.csv"
        fold_val_csv   = f"reg_val_{repeat_idx}_{fold_counter}.csv"

        df_fold_trn.to_csv(fold_train_csv, index=False)
        df_fold_val.to_csv(fold_val_csv, index=False)

        print("Wrote:", fold_train_csv, "|", fold_val_csv)
        fold_counter += 1

Wrote: reg_trn_0_0.csv | reg_val_0_0.csv
Wrote: reg_trn_0_1.csv | reg_val_0_1.csv
Wrote: reg_trn_0_2.csv | reg_val_0_2.csv
Wrote: reg_trn_0_3.csv | reg_val_0_3.csv
Wrote: reg_trn_0_4.csv | reg_val_0_4.csv
Wrote: reg_trn_1_0.csv | reg_val_1_0.csv
Wrote: reg_trn_1_1.csv | reg_val_1_1.csv
Wrote: reg_trn_1_2.csv | reg_val_1_2.csv
Wrote: reg_trn_1_3.csv | reg_val_1_3.csv
Wrote: reg_trn_1_4.csv | reg_val_1_4.csv
Wrote: reg_trn_2_0.csv | reg_val_2_0.csv
Wrote: reg_trn_2_1.csv | reg_val_2_1.csv
Wrote: reg_trn_2_2.csv | reg_val_2_2.csv
Wrote: reg_trn_2_3.csv | reg_val_2_3.csv
Wrote: reg_trn_2_4.csv | reg_val_2_4.csv
